# QMC.jl for Lebesgue Integration

This notebook adapts QMCPy's `lebesgue_integration.ipynb` to the Julia `QMC.jl` APIs.

Original QMCPy demo: [`QMCPy/demos/lebesgue_integration.ipynb`](../../QMCPy/demos/lebesgue_integration.ipynb)

QMC.jl preserves the same four sample problems as QMCPy while constructing `Lebesgue` directly from a discrete distribution instead of wrapping another true measure, so the equivalent volume and Gaussian weights are written explicitly in the Julia integrands below.

Parity note: QMCPy uses `replications=32` in the Halton and digital-net examples. The Julia notebook keeps smaller replication counts so the checked-in demo remains lightweight while preserving the same four-problem structure and the shared digital-net `1e-3` tolerance target.


In [1]:
using QMC
using Printf


## Sample Problem 1

$$y = \int_{[0,2]} x^2 \, \mathrm{d}x = 2\int_{[0,2]} \frac{x^2}{2} \, \mathrm{d}x$$

The first expression is naturally a Lebesgue integral on `[0,2]`, while the second can be viewed as an expectation under the uniform measure on the same interval.


In [2]:
abs_tol = 0.01
dim = 1
a = 0.0
b = 2.0
true_value = 8.0 / 3.0


2.6666666666666665

In [3]:
# Lebesgue measure
tm = Lebesgue(Halton(dim; seed=7, replications=4); lower_bound=a, upper_bound=b)
integrand = CustomFun(tm, x -> tm.volume .* vec(sum(x .^ 2, dims=2)))
solution = integrate(CubMCCLT(integrand; abs_tol=abs_tol)).solution
@printf("Lebesgue measure: y = %.3f
", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Lebesgue measure: y = 2.667


false

In [4]:
# Uniform measure
tm = Uniform(Halton(dim; seed=7, replications=4); lower_bound=a, upper_bound=b)
integrand = CustomFun(tm, x -> (b - a) .* vec(sum(x .^ 2, dims=2)))
solution = integrate(CubMCCLT(integrand; abs_tol=abs_tol)).solution
@printf("Uniform measure:  y = %.3f
", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Uniform measure:  y = 2.667


false

## Sample Problem 2

$$y = \int_{[a,b]^d} \lVert x \rVert_2^2 \, \mathrm{d}x = \prod_{i=1}^d (b_i-a_i)\int_{[a,b]^d} \lVert x \rVert_2^2 \; \left[\prod_{i=1}^d (b_i-a_i)\right]^{-1} \, \mathrm{d}x$$

Again, the same quantity can be written either as a Lebesgue integral or as a uniform expectation with an explicit volume factor.


In [5]:
abs_tol = 1e-3
dim = 2
a = [1.0, 2.0]
b = [2.0, 4.0]
true_value = ((a[1]^3 - b[1]^3) * (a[2] - b[2]) + (a[1] - b[1]) * (a[2]^3 - b[2]^3)) / 3
@printf("Answer = %.5f
", true_value)


Answer = 23.33333


In [6]:
# Lebesgue measure
tm = Lebesgue(DigitalNetB2(dim; seed=7, randomize="LMS_DS"); lower_bound=a, upper_bound=b)
integrand = CustomFun(tm, x -> tm.volume .* vec(sum(x .^ 2, dims=2)))
solution = integrate(CubMCCLT(integrand; abs_tol=abs_tol)).solution
@printf("Lebesgue measure: y = %.5f
", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Lebesgue measure: y = 23.33333


false

In [7]:
# Uniform measure
tm = Uniform(DigitalNetB2(dim; seed=17, randomize="LMS_DS"); lower_bound=a, upper_bound=b)
integrand = CustomFun(tm, x -> prod(b .- a) .* vec(sum(x .^ 2, dims=2)))
solution = integrate(CubMCCLT(integrand; abs_tol=abs_tol)).solution
@printf("Uniform measure:  y = %.5f
", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Uniform measure:  y = 23.33333


false

## Sample Problem 3

Integral that cannot be expressed in terms of elementary special functions:

$$y = \int_{[a,b]} \frac{\sin(x)}{\log(x)} \, \mathrm{d}x$$

As in the QMCPy demo, the point is to show that the Lebesgue formulation is still easy to handle numerically.


In [8]:
abs_tol = 1e-4
dim = 1
a = 3.0
b = 5.0
true_value = -0.87961


-0.87961

In [9]:
# Lebesgue measure
tm = Lebesgue(Lattice(dim; randomize=true, seed=7); lower_bound=a, upper_bound=b)
integrand = CustomFun(tm, x -> tm.volume .* vec(sin.(x) ./ log.(x)))
solution = integrate(CubQMCLatticeG(integrand; abs_tol=abs_tol)).solution
@printf("Lebesgue measure: y = %.3f
", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Lebesgue measure: y = -0.880


false

## Sample Problem 4

Integral over $\mathbb{R}^d$:

$$y = \int_{\mathbb{R}^2} e^{-\lVert x \rVert_2^2} \, \mathrm{d}x = \pi$$

QMCPy writes this as `Lebesgue(Gaussian(...))`. QMC.jl does not yet expose that wrapper directly, so we integrate with respect to a Gaussian measure and include the Lebesgue-to-Gaussian weight explicitly in the integrand.


In [10]:
abs_tol = 0.1
dim = 2
true_value = π


π = 3.1415926535897...

In [11]:
tm = Gaussian(Lattice(dim; randomize=true, seed=7); mean=0.0, covariance=1.0)
integrand = CustomFun(tm, x -> vec((2π)^(dim / 2) .* prod(exp.(-x .^ 2 ./ 2), dims=2)))
solution = integrate(CubQMCLatticeG(integrand; abs_tol=abs_tol)).solution
@printf("Gaussian measure with Lebesgue weight: y = %.3f
", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Gaussian measure with Lebesgue weight: y = 3.142


false